# 25회차 실습: KNN 얼굴 맞히기와 로지스틱 회귀

위에서부터 한 칸씩 `Shift + Enter`로 실행하며 따라와요. 칸 위 **나와야 하는 것**과 내 결과가 같으면 맞게 따라온 거예요.

- 프로젝트: 22회차의 `~/dev/my-pandas` (pandas, scikit-learn, seaborn, matplotlib이 들어 있어요)
- 얼굴 사진은 24회차에 받은 `datasets` 폴더에서 읽어요. 없으면 첫 실행 때 약 1.4MB를 받아요.
- 수업 코드의 `plt.axis="off"`는 함수를 덮어쓰는 줄이라 모두 `plt.axis("off")`로 고쳐 뒀어요.

## 1. 준비: 도구와 얼굴 사진 불러오기

**나와야 하는 것:** `X: (400, 4096)`과 `y: (400,)`. 사진 400장이 한 장에 숫자 4096개씩, 정답(사람 번호)이 400개예요.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_olivetti_faces
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix

# 섞이지 않은 원본(0번 10장, ..., 39번 10장)
faces = fetch_olivetti_faces(data_home="./datasets")

# 사진 한 장(64 × 64)을 한 줄로 편 데이터
X = faces.data
# 정답: 몇 번 사람인가
y = faces.target

print("X:", X.shape)
print("y:", y.shape)

## 2. 40명 얼굴 한눈에

**나와야 하는 것:** `(64, 64)`. 사진 한 장은 64 × 64 표예요.

In [ ]:
faces.images[0].shape

**나와야 하는 것:** 0번부터 39번까지 제목이 붙은 흑백 얼굴 40장이 4줄 10칸으로.

In [ ]:
fig, axes = plt.subplots(4, 10, figsize=(15, 8))

# axes.flat: 4 × 10 칸을 한 줄로 늘어놓기
for person, ax in enumerate(axes.flat):
    # 그 사람의 사진 10장 중 첫 장
    first = faces.images[faces.target == person][0]
    ax.imshow(first, cmap="gray", vmin=0, vmax=1)
    ax.set_title(person, fontsize=10)
    ax.axis("off")

plt.show()

**해 보기:** 아래 칸의 `#`을 지우고 실행하면 `TypeError: 'str' object is not callable`이 나요. `plt.axis="off"`가 `plt.axis` 함수를 글자로 덮어썼기 때문이에요. 해 본 뒤에는 위쪽 **Restart**로 커널을 다시 켜고 1절부터 다시 실행해요.

In [ ]:
# plt.axis = "off"
# plt.axis("off")

## 3. KNN: 가장 가까운 사진 K장의 다수결

229번 사진 한 장을 빼 두고 나머지 399장으로 학습해, 229번이 누구인지 물어봐요.

**나와야 하는 것:** `399`. 229번만 빠졌어요.

In [ ]:
new_index = 229
new_photo = X[new_index]

# 229번만 False, 나머지 399장은 True
known = np.arange(400) != new_index
X_known, y_known = X[known], y[known]

print(known.sum())

**나와야 하는 것:**

```text
가장 가까운 이웃: [17 22 22]
이웃과의 거리: [6.07 6.26 6.44]
예측한 사람 : 22
실제 정답: 22
```

`K = 10`으로 바꿔 다시 실행하면 이웃 10장 중 17번이 4장, 22번이 3장이라 `예측한 사람 : 17`로 틀려요. 해 본 뒤에는 `K = 3`으로 되돌려 실행해요(4절이 이 결과를 써요).

In [ ]:
# 인접한 이웃 수
K = 3

model = KNeighborsClassifier(n_neighbors=K)
model.fit(X_known, y_known)

# 가장 가까운 이웃 K개의 거리와 X_known 안의 자리
dist, idx = model.kneighbors([new_photo], n_neighbors=K)

print("가장 가까운 이웃:", y_known[idx[0]])
print("이웃과의 거리:", dist[0].round(2))

pred = model.predict([new_photo])
print("예측한 사람 :", pred[0])
print("실제 정답:", y[new_index])

거리는 같은 칸끼리 밝기 차이를 제곱해 4096칸 모두 더하고 제곱근을 씌운 값이에요(유클리드 거리).

**나와야 하는 것:** `[6.07 6.26 6.44]`. 위 `kneighbors`의 거리와 같아요.

In [ ]:
diff = X_known - new_photo
d = np.sqrt((diff ** 2).sum(axis=1))
print(np.sort(d)[:3].round(2))

## 4. 이웃 사진 그리기

`X`의 한 줄은 숫자 4096개라, `reshape(64, 64)`로 다시 접어야 사진으로 그릴 수 있어요.

**나와야 하는 것:** `Person 22`라는 제목의 작은 얼굴 한 장(229번 사진).

In [ ]:
plt.figure(figsize=(1, 1))
plt.imshow(new_photo.reshape(64, 64), cmap="gray")
plt.title(f"Person {y[new_index]}")
plt.axis("off")
plt.show()

**나와야 하는 것:** `Person 17` 사진 한 장. 가장 가까운 이웃이 하필 다른 사람이에요.

In [ ]:
# 가장 가까운 이웃의 X_known 안 자리
j = idx[0][0]

plt.figure(figsize=(1, 1))
plt.imshow(X_known[j].reshape(64, 64), cmap="gray")
plt.title(f"Person {y_known[j]}")
plt.axis("off")
plt.show()

**나와야 하는 것:** 가까운 순서대로 `Person 17`, `Person 22`, `Person 22` 작은 사진 세 장.

In [ ]:
for i in range(K):
    j = idx[0][i]

    plt.figure(figsize=(1, 1))
    plt.imshow(X_known[j].reshape(64, 64), cmap="gray")
    plt.title(f"Person {y_known[j]}")
    plt.axis("off")
    plt.show()

## 5. KNN 채점: 처음 보는 100장으로

**나와야 하는 것:** `(300, 4096) (100, 4096)`과 `예측 정확도: 87.00%`.

In [ ]:
# 전체 사진을 학습용과 테스트용으로 분리
X_train, X_test, y_train, y_test = train_test_split(
    faces.data,
    faces.target,
    test_size=0.25,
    random_state=42,
    # 사람마다 시험용 사진 비율을 고르게
    stratify=faces.target
)
print(X_train.shape, X_test.shape)

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)

# 처음 보는 테스트 사진으로 정확도 평가
accuracy = knn.score(X_test, y_test)
print(f"예측 정확도: {accuracy:.2%}")

**나와야 하는 것:** `stratify`를 주면 사람마다 시험용 사진이 2~3장이에요. 첫 줄이 `[2 3 3 3 2 3 2 3 2 2 3 3 2 2 3 2 2 2 3 2 3 3 3 3 2 2 2 2 3 3 3 2 3 2 2 3 3`로 시작해요.

In [ ]:
# 사람 번호별 시험용 사진 장수
print(np.bincount(y_test, minlength=40))

## 6. 로지스틱 회귀: 합격, 불합격 맞히기

**나와야 하는 것:**

```text
학생 10 명, X : 3개
   score  passed
0     47   False
1     58   False
2     60    True
3     76    True
4     87    True
```

In [ ]:
df = pd.DataFrame({
    "study_hours":    [1, 2, 3, 4, 5, 2, 3, 5, 4, 1],
    "sleep_hours":    [6, 8, 5, 7, 8, 5, 8, 6, 6, 8],
    "practice_count": [10, 15, 20, 25, 30, 12, 28, 22, 32, 8],
    "score":          [47, 58, 60, 76, 87, 49, 73, 77, 79, 52],
})

X = df[["study_hours", "sleep_hours", "practice_count"]]
# 60점 이상이면 True(합격)
y = df["score"] >= 60

print(f"학생 {len(X)} 명, X : {X.shape[1]}개")
print(df[["score"]].assign(passed=y).head())

**나와야 하는 것:** `학습 7명, 테스트 3명`, `첫 학습 학생의 공부시간: 4 시간`, `표준화 값: 1.21`.

In [ ]:
# 합격, 불합격 비율을 고르게 나누기
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"학습 {len(X_train)}명, 테스트 {len(X_test)}명")
# iloc[행, 열]: 자리 번호로 꺼내기
print(f"첫 학습 학생의 공부시간: {X_train.iloc[0, 0]} 시간")
print(f"표준화 값: {X_train_scaled[0, 0]:.2f}")

**나와야 하는 것:** `클래스 : [False, True]`와 `가중치 : [ 0.873 -0.37   0.877]`.

In [ ]:
# 표준화된 학습 데이터 + 정답
model = LogisticRegression(max_iter=1000)
model.fit(X_train_scaled, y_train)

print("클래스 :", model.classes_.tolist())
print("가중치 :", model.coef_[0].round(3))

## 7. 새 학생 합격 확률과 채점

**나와야 하는 것:** `예측 결과: 합격`, `합격 확률: 92.58%`, 그리고 `[[0.0742 0.9258]]` (불합격 확률, 합격 확률).

In [ ]:
new = pd.DataFrame([[4, 7, 30]], columns=X.columns)
new_scaled = scaler.transform(new)

predict = model.predict(new_scaled)[0]
# [0, 1]: 첫 학생의 True(합격) 확률
pass_proba = model.predict_proba(new_scaled)[0, 1]

print("예측 결과:", "합격" if predict else "불합격")
print(f"합격 확률: {pass_proba:.2%}")
print(model.predict_proba(new_scaled).round(4))

**나와야 하는 것:**

```text
실제 정답:  [False, True, True]
예측 결과:  [False, True, True]
정확도: 100.0000%
[[1 0]
 [0 2]]
```

In [ ]:
# 학습에 쓰지 않은 학생 3명으로 예측, 평가
test_pred = model.predict(X_test_scaled)

print("실제 정답: ", y_test.tolist())
print("예측 결과: ", test_pred.tolist())
print(f"정확도: {accuracy_score(y_test, test_pred):.4%}")
print(confusion_matrix(y_test, test_pred, labels=[False, True]))

**나와야 하는 것:** 2 × 2 파란 칸. 왼쪽 위 `1`, 오른쪽 아래 `2`(진한 파랑), 나머지 `0`. 가로축 Predicted, 세로축 Actual.

In [ ]:
cm = confusion_matrix(y_test, test_pred, labels=[False, True])

plt.figure(figsize=(4, 3))
sns.heatmap(
    cm,
    annot=True,
    cmap="Blues",
    xticklabels=["Fail", "Pass"],
    yticklabels=["Fail", "Pass"]
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

## 혼자 치기

위 칸을 모두 실행한 뒤에 풀어요(`faces`, `df`, `X`, `scaler`, `model`을 그대로 써요). 정답은 맨 아래에 있어요.

**문제 1.** 3절처럼 229번을 뺀 399장으로, 이번에는 `K = 5`로 학습해 229번 사진의 예측을 찍어 보세요.

In [ ]:
# 여기에 써 보세요

**문제 2.** 395번 사진을 빼 두고 나머지 399장으로 `K = 3` KNN을 학습해, 이웃 3장의 사람 번호와 예측, 실제 정답을 찍어 보세요. 맞혔나요?

In [ ]:
# 여기에 써 보세요

**문제 3.** 5절과 똑같이 나눈 얼굴 사진으로 `K = 1` KNN을 학습해 정확도를 `:.2%`로 찍어 보세요. K = 5의 87%와 비교해 보세요.

In [ ]:
# 여기에 써 보세요

**문제 4.** 합격 기준을 70점으로 올리면 10명 중 합격자는 몇 명인지 찍어 보세요.

In [ ]:
# 여기에 써 보세요

**문제 5.** 6절의 `scaler`와 `model`로 공부 2시간, 수면 6시간, 문제 12개인 학생의 합격 확률을 `:.2%`로, 예측 결과(True, False)도 함께 찍어 보세요.

In [ ]:
# 여기에 써 보세요

## 정답

먼저 풀어 본 뒤에 봐요.

문제 1

```python
known = np.arange(400) != 229
five = KNeighborsClassifier(n_neighbors=5)
five.fit(faces.data[known], faces.target[known])
print(five.predict([faces.data[229]])[0])
# 22
```

문제 2

```python
known = np.arange(400) != 395
three = KNeighborsClassifier(n_neighbors=3)
three.fit(faces.data[known], faces.target[known])

dist, idx = three.kneighbors([faces.data[395]])
print(faces.target[known][idx[0]])
print(three.predict([faces.data[395]])[0])
print(faces.target[395])
# [39  4 29]
# 4
# 39
# 이웃 세 장이 모두 다른 사람이라 동점이고, 번호가 작은 4번을 골라 틀려요.
```

문제 3

```python
X_tr, X_te, y_tr, y_te = train_test_split(
    faces.data,
    faces.target,
    test_size=0.25,
    random_state=42,
    stratify=faces.target
)
one = KNeighborsClassifier(n_neighbors=1)
one.fit(X_tr, y_tr)
print(f"{one.score(X_te, y_te):.2%}")
# 95.00%
```

문제 4

```python
print((df["score"] >= 70).sum())
# 5
```

문제 5

```python
student = pd.DataFrame([[2, 6, 12]], columns=X.columns)
student_scaled = scaler.transform(student)
print(f"{model.predict_proba(student_scaled)[0, 1]:.2%}")
print(model.predict(student_scaled)[0])
# 37.91%
# False
```